# Loading the data:

In [7]:
# load the cleaned dataset
#from ReusableFunc import load_clean_data


#PrData = load_clean_data(r"C:\Users\Hi\Desktop\Risk-Analysis\data\processed\ProData.csv")

#print("Data loaded successfully!")
import pandas as pd
clean_data=pd.read_csv(r"C:\Users\Hi\Desktop\Risk-Analysis\data\processed\ProcessedData.csv")
print("Data Loaded Successfully")

Data Loaded Successfully


In [8]:
clean_data.isnull().sum()

Patient_ID             0
Age                    0
Gender                 0
Region                 0
Insurance_Type         0
Admission_Type         0
Hospital_Department    0
Length_of_Stay         0
Previous_Admissions    0
Previous_ER_Visits     0
Diabetes               0
Hypertension           0
Heart_Disease          0
Medication_Count       0
Lab_Test_Count         0
Average_Glucose        0
Systolic_BP            0
Discharge_Type         0
Followup_Scheduled     0
Followup_Attended      0
Treatment_Cost         0
Satisfaction_Score     0
Readmitted_30_Days     0
dtype: int64

# Statistical Analysis

# 1. What is the typical patients Age, Length of stay and Treatment cost?

In [9]:
clean_data[['Age', 'Length_of_Stay', 'Treatment_Cost']].describe().T

,count,mean,std,min,25%,50%,75%,max
Age,500.0,53.990,20.743831,18.0,37.0,54.0,72.00,90.0
Length_of_Stay,500.0,4.678,4.258472,1.0,2.0,4.0,6.00,60.0
Treatment_Cost,500.0,85775.874,32339.752884,17712.0,62532.0,81396.5,104412.25,193348.0


Descriptive analysis was conducted to understand the distribution of key numerical variables. The average patient age was 53.99 years (SD = 20.74), with ages ranging from 18 to 90 years. 

Hospital length of stay averaged 4.68 days (SD = 4.25), with a median of 4 days and a range of 1 to 60 days. The maximum length of stay of 60 days appears substantially higher than the typical stay and should be investigated as a potential outlier. 

Treatment costs averaged approximately KSh 85,775 and a standard deviation of KSh 32,339, indicating considerable variation in treatment expenditure.

# 2. Is Readmission associated with Diabetes, gender, Admission type and discharge type?

In [10]:
import pandas as pd

# Create a crosstab to analyze the relationship between Diabetes and Readmission within 30 days
pd.crosstab(
    clean_data['Diabetes'],
    clean_data['Readmitted_30_Days'],
    normalize='index'
).mul(100).round(2)

# Create a crosstab to analyze the relationship between Gender and Readmission within 30 days
#pd.crosstab(
   # PrData['Gender'],
    #PrData['Readmitted_30_Days'],
   # normalize='index'
#).mul(100).round(2) 

Readmitted_30_Days,No,Yes
Diabetes,,
No,79.51,20.49
Yes,66.42,33.58


Diabetes vs 30-Day Readmission

The cross-tabulation indicates a difference in 30-day readmission between patients with and without diabetes. Among patients without diabetes, 20.49% were readmitted within 30 days, while 79.51% were not readmitted. In contrast, among patients with diabetes, 33.58% were readmitted and 66.42% were not readmitted. This suggests that patients with diabetes had a higher proportion of 30-day readmissions compared with patients without diabetes.

Key finding
Diabetes	Not readmitted	Readmitted
No	79.51%	20.49%
Yes	66.42%	33.58%

So, readmission was about 13.09 percentage points higher among patients with diabetes (33.58% vs 20.49%).

However, this is only a descriptive finding. To determine whether the relationship between diabetes and readmission is statistically significant, you should follow it with a chi-square test of independence.

# 3. Chi-square test

The chi-square test is appropriate when you want to determine whether two categorical variables are statistically associated.

For example:

Diabetes vs 30-day readmission

In [11]:
from scipy.stats import chi2_contingency

table = pd.crosstab(
    clean_data['Diabetes'],
    clean_data['Readmitted_30_Days']
)

chi2, p, dof, expected = chi2_contingency(table)

print("Chi-square:", chi2)
print("p-value:", p)

Chi-square: 8.511176680589005
p-value: 0.003529717454583244


A chi-square test of independence was conducted to assess the association between diabetes status and 30-day hospital readmission. The test showed a statistically significant association between diabetes and 30-day readmission (χ² = 8.511, p = 0.003). Patients with diabetes had a higher readmission rate (33.58%) compared with patients without diabetes (20.49%). Therefore, diabetes status appears to be associated with 30-day readmission in this dataset.

In conclusion: Diabetes was significantly associated with 30-day readmission

# 5. Correlation Analysis:

Does length of stay increase as treatment cost increase?

In [12]:
clean_data[['Length_of_Stay', 'Treatment_Cost']].corr()


,Length_of_Stay,Treatment_Cost
Length_of_Stay,1.000000,0.640875
Treatment_Cost,0.640875,1.000000


There was a moderately strong positive correlation between Length of Stay and Treatment Cost (r = 0.641). This indicates that patients with longer hospital stays tended to incur higher treatment costs. The finding suggests that length of stay may be an important factor associated with treatment expenditure. However, correlation does not imply causation, and other patient and treatment-related factors may also influence treatment costs.

# group comparison:


In [13]:
# Summary statistics by readmission status

summary = clean_data.groupby('Readmitted_30_Days')[
    ['Length_of_Stay', 'Treatment_Cost']
].mean()

print(summary.round(2))

                    Length_of_Stay  Treatment_Cost
Readmitted_30_Days                                
No                            4.52        84564.54
Yes                           5.17        89611.76


Interpretation

Patients who were readmitted within 30 days had an average hospital stay of 5.18 days, compared with 4.52 days among patients who were not readmitted. This represents a difference of approximately 0.66 days.

Similarly, patients who were readmitted had a higher average treatment cost of approximately KSh 89,611, compared with KSh 84,564 among patients who were not readmitted. This represents a difference of approximately KSh 5,047.

Overall finding

The descriptive analysis suggests that patients who experienced 30-day readmission tended to have longer hospital stays and higher treatment costs than those who were not readmitted. This may indicate that length of stay and treatment cost are associated with readmission outcomes. However, these differences alone do not establish statistical significance or causation.

To confirm this we need to test whether these differences are statistically significant using a t-test (or Mann–Whitney U test) for Length of Stay and Treatment Cost between the two readmission groups.

# Independent Samples t-test:
Does Length of Stay and Treatment Cost differ significantly between patients who were readmitted and those who were not?

use an independent samples t-test. If the data are not normally distributed, use the Mann–Whitney U test.

# Normality Test:

In [14]:
from scipy.stats import shapiro

# Length of Stay
los_no = clean_data.loc[clean_data['Readmitted_30_Days'] == 'No', 'Length_of_Stay'].dropna()
los_yes = clean_data.loc[clean_data['Readmitted_30_Days'] == 'Yes', 'Length_of_Stay'].dropna()

print("Length of Stay - Not Readmitted")
print(shapiro(los_no))

print("\nLength of Stay - Readmitted")
print(shapiro(los_yes))


# Treatment Cost
cost_no = clean_data.loc[clean_data['Readmitted_30_Days'] == 'No', 'Treatment_Cost'].dropna()
cost_yes = clean_data.loc[clean_data['Readmitted_30_Days'] == 'Yes', 'Treatment_Cost'].dropna()

print("\nTreatment Cost - Not Readmitted")
print(shapiro(cost_no))

print("\nTreatment Cost - Readmitted")
print(shapiro(cost_yes))

Length of Stay - Not Readmitted
ShapiroResult(statistic=np.float64(0.5941608598097735), pvalue=np.float64(7.336756265094407e-29))

Length of Stay - Readmitted
ShapiroResult(statistic=np.float64(0.636302757351092), pvalue=np.float64(8.242366149913809e-16))

Treatment Cost - Not Readmitted
ShapiroResult(statistic=np.float64(0.9724184082454735), pvalue=np.float64(1.2798337811022448e-06))

Treatment Cost - Readmitted
ShapiroResult(statistic=np.float64(0.948926395331642), pvalue=np.float64(0.00018038109788834868))


Not normally distributed , use mann-whitney-u

In [15]:
from scipy.stats import mannwhitneyu

# Length of Stay
u_los, p_los_mw = mannwhitneyu(
    los_no,
    los_yes,
    alternative='two-sided'
)

print("Length of Stay - Mann-Whitney U")
print("U-statistic:", u_los)
print("p-value:", p_los_mw)


# Treatment Cost
u_cost, p_cost_mw = mannwhitneyu(
    cost_no,
    cost_yes,
    alternative='two-sided'
)

print("\nTreatment Cost - Mann-Whitney U")
print("U-statistic:", u_cost)
print("p-value:", p_cost_mw)

Length of Stay - Mann-Whitney U
U-statistic: 21371.5
p-value: 0.2965259870353891

Treatment Cost - Mann-Whitney U
U-statistic: 21658.0
p-value: 0.40806463635332424


A Mann–Whitney U test was conducted to assess whether Length of Stay and Treatment Cost differed significantly between patients who were readmitted within 30 days and those who were not. For Length of Stay, the test produced a U-statistic of 21,371 and a p-value of 0.296. Since the p-value was greater than 0.05, there was no statistically significant difference in Length of Stay between the two groups.

For Treatment Cost, the Mann–Whitney U test produced a U-statistic of 21,658 and a p-value of 0.408. Since the p-value was also greater than 0.05, there was no statistically significant difference in Treatment Cost between readmitted and non-readmitted patients.

Although readmitted patients had a higher average Length of Stay (5.18 days versus 4.52 days) and higher average Treatment Cost (KSh 89,891.17 versus KSh 84,601.39), these observed differences were not statistically significant at the 5% significance level. 

# Conclusion:
Therefore, the analysis does not provide sufficient statistical evidence that Length of Stay or Treatment Cost is associated with 30-day readmission in this dataset.


# Test age,previous admission against 30 day readmission:


Normality test

In [17]:
from scipy.stats import shapiro

# age
age_no = clean_data.loc[clean_data['Readmitted_30_Days'] == 'No', 'Age'].dropna()
age_yes = clean_data.loc[clean_data['Readmitted_30_Days'] == 'Yes', 'Age'].dropna()

print("Age - Not Readmitted")
print(shapiro(age_no))

print("\nAge - Readmitted")
print(shapiro(age_yes))


# previous readmission
prev_no = clean_data.loc[clean_data['Readmitted_30_Days'] == 'No', 'Previous_Admissions'].dropna()
prev_yes = clean_data.loc[clean_data['Readmitted_30_Days'] == 'Yes', 'Previous_Admissions'].dropna()

print("\nPrevious Admissions - Not Readmitted")
print(shapiro(prev_no))

print("\nPrevious Admissions - Readmitted")
print(shapiro(prev_yes))

Age - Not Readmitted
ShapiroResult(statistic=np.float64(0.9576137844414386), pvalue=np.float64(5.224497793337002e-09))

Age - Readmitted
ShapiroResult(statistic=np.float64(0.9467919343610119), pvalue=np.float64(0.00012661728953345828))

Previous Admissions - Not Readmitted
ShapiroResult(statistic=np.float64(0.8411973404966842), pvalue=np.float64(4.0488367715397124e-19))

Previous Admissions - Readmitted
ShapiroResult(statistic=np.float64(0.8842520535085757), pvalue=np.float64(3.3279052759050966e-08))
